<a href="https://colab.research.google.com/github/Soonly-T/cv-aupp/blob/main/Copy_of_Computer_Vision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics

In [ ]:
import os
import json
import random
import shutil
import cv2

def read_and_process_annotations(json_file, original_images_path):
    """
    Reads annotations from a JSON file, extracts relevant information,
    and returns a dictionary mapping filenames to annotations.

    Handles different JSON structures for VIA annotations.
    """
    with open(json_file, 'r') as f:
        data = json.load(f)

    # Check if JSON structure contains 'file' key (older VIA format)
    if 'file' in data:
        fid_to_filename = {v['fid']: v['fname'] for v in data['file'].values()}
        annotations = {}
        for meta in data['metadata'].values():
            vid = meta['vid']
            fname = fid_to_filename.get(vid)
            if not fname:
                continue

            xy = meta.get('xy', [])
            av = meta.get('av', {})
            if not xy or not av:
                continue

            if fname not in annotations:
                annotations[fname] = []

            x = xy[1]
            y = xy[2]
            w = xy[3]
            h = xy[4]
            class_id = int(av["1"])

            annotations[fname].append([class_id, x, y, w, h])

        return annotations

    # If 'file' key is not present, assume newer VIA format
    else:
        annotations = {}
        for filename, data in data.items():  # Iterate through filenames in the JSON
            if not filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue  # Skip if not an image file

            regions = data.get('regions', [])
            if not regions:
                continue  # Skip if no regions (annotations)

            annotations[filename] = []  # Initialize list for this filename
            for region in regions:
                shape_attributes = region['shape_attributes']
                region_attributes = region['region_attributes']

                x = shape_attributes['x']
                y = shape_attributes['y']
                w = shape_attributes['width']
                h = shape_attributes['height']
                # Assuming class ID is in 'class_id' key of region_attributes
                class_id = int(region_attributes.get('class_id', 0))
                annotations[filename].append([class_id, x, y, w, h])
        return annotations

# --- Example Usage ---

# Define paths for your JSON files and image directories
json_file1 = '/content/drive/MyDrive/Data/annotation/annotation1/4eefd9e3_27Apr2025_13h18m16s.json'
original_images_path1 = '/content/drive/MyDrive/Data/images/images1'

json_file2 = '/content/drive/MyDrive/Data/annotation/annotation2/via_project_28Apr2025_04h16m43s.json'
original_images_path2 = '/content/drive/MyDrive/Data/images/images2'

#json_file3 = '/content/drive/MyDrive/Data/annotation/annotation3/602-901Annotation.json'
#original_images_path3 = '/content/drive/MyDrive/Data/images/images3'

# Read and process annotations for each pair
annotations1 = read_and_process_annotations(json_file1, original_images_path1)
print(f"Found annotations for {len(annotations1)} images")

annotations2 = read_and_process_annotations(json_file2, original_images_path2)
print(f"Found annotations for {len(annotations2)} images")

#annotations3 = read_and_process_annotations(json_file3, original_images_path3)
#print(f"Found annotations for {len(annotations3)} images")

# Merge annotations (if there are common filenames, you might need to handle them carefully)
all_annotations = {**annotations1, **annotations2}
print(f"Found annotations for {len(all_annotations)} images")

# Build a lookup for original images path
filename_to_folder = {}

for fname in annotations1.keys():
    filename_to_folder[fname] = original_images_path1

for fname in annotations2.keys():
    filename_to_folder[fname] = original_images_path2

Found annotations for 300 images
Found annotations for 235 images
Found annotations for 532 images


In [ ]:
# Output dataset path
dataset_path = '/content/dataset'
images_train_path = os.path.join(dataset_path, 'images/train')
labels_train_path = os.path.join(dataset_path, 'labels/train')
images_test_path = os.path.join(dataset_path, 'images/test')
labels_test_path = os.path.join(dataset_path, 'labels/test')
images_val_path = os.path.join(dataset_path, 'images/val')
labels_val_path = os.path.join(dataset_path, 'labels/val')
all_images = os.path.join(dataset_path, 'images/all')
all_labels = os.path.join(dataset_path, 'labels/all')

# Create folders
os.makedirs(images_train_path, exist_ok=True)
os.makedirs(labels_train_path, exist_ok=True)
os.makedirs(images_val_path, exist_ok=True)
os.makedirs(labels_val_path, exist_ok=True)
os.makedirs(images_test_path, exist_ok=True)
os.makedirs(labels_test_path, exist_ok=True)
os.makedirs(all_images, exist_ok=True)
os.makedirs(all_labels, exist_ok=True)

# Split ratio
train_split = 0.8
test_split = 0.1

vehicle_classes = {
    "0": "Car",
    "1": "Bus",
    "2": "Tram",
    "3": "Taxi",
    "4": "Police Car",
    "5": "Ambulance",
    "6": "Delivery Van",
    "7": "Semi Truck",
    "8": "Road Maintenance",
    "9": "Park Maintenance",
    "10": "Hearse",
    "11": "City Car",
    "12": "Garbage Truck",
    "13": "Post Van",
    "14": "Motorcycle",
    "15": "Trailer"
}

vehicle_type_priority = {
    "Car": 5, # Includes City Car, Motorcycle, Road Maintenance, Park Maintenance, Hearse, Muscle Car, Taxi, Hatchback/Sedan, Van
    "Bus": 80,
    "Tram": 240,
    "Police Car": float('inf'), # High priority for emergency vehicles
    "Ambulance": float('inf'), # High priority for emergency vehicles
    "Delivery Van": 4,
    "Semi Truck": 14.5,
    "Post Van": 2,
    "Garbage Truck": 15,
    "Trailer": 0,
    "Motorcycle": 2, # Explicitly adding based on the image grouping
    "Road Maintenance": 2, # Explicitly adding based on the image grouping
    "Park Maintenance": 2, # Explicitly adding based on the image grouping
    "Hearse": 2, # Explicitly adding based on the image grouping
    "Taxi": 5, # Explicitly adding based on the new grouping


}

# Split into train and val
all_files = list(all_annotations.keys())

split_idx = int(len(all_files) * train_split)
split_idx2 = int(len(all_files) * (train_split + test_split))

train_files = all_files[:split_idx]
test_files = all_files[split_idx:split_idx2]
val_files = all_files[split_idx2:]

print(f"Train: {len(train_files)}")
print(f"Test: {len(test_files)}")
print(f"Val: {len(val_files)}")

# Save images and YOLO labels
def save_yolo_labels(filename, labels, output_folder, img_width, img_height):
    label_file = os.path.join(output_folder, filename.replace('.jpg', '.txt'))
    with open(label_file, 'w') as f:
        for cls, x, y, w, h in labels:
            # Convert to YOLO format: center_x center_y width height (normalized)
            center_x = (x + w/2) / img_width
            center_y = (y + h/2) / img_height
            w /= img_width
            h /= img_height
            f.write(f"{cls} {center_x} {center_y} {w} {h}\n")

def process_files(file_list, img_out_dir, label_out_dir, filename_to_folder):
    for fname in file_list:
        src_folder = filename_to_folder.get(fname)
        if src_folder is None:
            print(f"Warning: No source folder found for {fname}")
            continue

        src_img = os.path.join(src_folder, fname)
        dst_img = os.path.join(img_out_dir, fname)

        if not os.path.exists(src_img):
            print(f"Warning: Missing {src_img}")
            continue

        # Copy image
        shutil.copyfile(src_img, dst_img)

        # Get image size
        img = cv2.imread(src_img)
        if img is None:
            print(f"Warning: Could not read image {src_img}")
            continue
        h, w = img.shape[:2]

        # Save label
        save_yolo_labels(fname, all_annotations[fname], label_out_dir, w, h)

# Update calls to process_files
process_files(train_files, images_train_path, labels_train_path, filename_to_folder)
process_files(val_files, images_val_path, labels_val_path, filename_to_folder)
process_files(test_files, images_test_path, labels_test_path, filename_to_folder)
process_files(all_files, all_images, all_labels, filename_to_folder)


# Create dataset.yaml
dataset_yaml = f"""path: {dataset_path}
all: images/all
train: images/train
val: images/val
test: images/test

nc: {len(vehicle_classes)}
names: {list(vehicle_classes.values())}
"""

with open(os.path.join(dataset_path, 'dataset.yaml'), 'w') as f:
    f.write(dataset_yaml)

print("✅ Dataset is ready!")

Train: 425
Test: 53
Val: 54
✅ Dataset is ready!


In [ ]:
from ultralytics import YOLO

base_model = YOLO('yolov8n.pt')  # (small, fast model)

base_model.train(
    data='/content/dataset/dataset.yaml',
    epochs=50,
    imgsz=640,
    batch=32
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


100%|██████████| 6.25M/6.25M [00:00<00:00, 74.9MB/s]


Ultralytics 8.3.119 🚀 Python-3.11.12 torch-2.6.0+cu124 CPU (Intel Xeon 2.20GHz)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/content/dataset/dataset.yaml, epochs=50, time=None, patience=100, batch=32, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_boxes=True, line_width=

100%|██████████| 755k/755k [00:00<00:00, 14.0MB/s]

Overriding model.yaml nc=80 with nc=16

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytic

...

       5/50         0G      1.466      2.246     0.9862         59        640: 100%|██████████| 14/14 [07:02<00:00, 30.15s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:17<00:00, 17.90s/it]

                   all         54        834     0.0448      0.489      0.264      0.165



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


       6/50         0G      1.563      2.112     0.9964        380        640:   7%|▋         | 1/14 [01:03<13:47, 63.66s/it]


KeyboardInterrupt: 

In [ ]:
results = base_model.val()

In [ ]:
base_model.save()

In [ ]:
import glob
import numpy as np
import os
from PIL import Image

# --- IoU and conversion utilities ---

def compute_iou(box1, box2):
    xA = max(box1[0], box2[0])
    yA = max(box1[1], box2[1])
    xB = min(box1[2], box2[2])
    yB = min(box1[3], box2[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    box1Area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2Area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    return interArea / float(box1Area + box2Area - interArea + 1e-6)

def yolo_to_bbox(yolo_box, img_w, img_h):
    _, x_center, y_center, w, h = yolo_box
    x_center *= img_w
    y_center *= img_h
    w *= img_w
    h *= img_h
    x1 = x_center - w / 2
    y1 = y_center - h / 2
    x2 = x_center + w / 2
    y2 = y_center + h / 2
    return [x1, y1, x2, y2]

In [ ]:
# --- Main Evaluation Script ---

def evaluate_iou_on_directory(model_path, image_dir):
    model = YOLO(model_path)
    image_paths = glob.glob(os.path.join(image_dir, "*.jpg"))

    all_ious = []
    for image_path in image_paths:
        label_path = image_path.replace('/images/', '/labels/').replace('.jpg', '.txt')
        if not os.path.exists(label_path):
            continue

        img = Image.open(image_path)
        w, h = img.size

        # Load ground truth boxes
        gt_boxes = []
        with open(label_path, 'r') as f:
            for line in f:
                yolo_box = list(map(float, line.strip().split()))
                gt_boxes.append(yolo_to_bbox(yolo_box, w, h))

        # Predict with model
        results = model(image_path)
        pred_boxes = results[0].boxes.xyxy.cpu().numpy() if results[0].boxes is not None else []

        for gt_box in gt_boxes:
            if len(pred_boxes) == 0:
                all_ious.append(0.0)
                continue
            ious = [compute_iou(gt_box, pred_box) for pred_box in pred_boxes]
            all_ious.append(max(ious))

    # Final average
    if all_ious:
        avg_iou = sum(all_ious) / len(all_ious)
        print(f"Evaluated {len(image_paths)} images.")
        print(f"Average IoU across dataset: {avg_iou:.4f}")
    else:
        print("No IoU values calculated. Check your labels and predictions.")

In [ ]:
# --- Evaluating Test Set ---
evaluate_iou_on_directory("/content/computer_vision_model.pt", "/content/dataset/images/test")

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/runs/detect/train/weights/last.pt')

model.train(
    data='/content/dataset/alldataset.yaml',
    epochs=30,
    imgsz=640,
    batch=32,
    lr0=0.0005,
)

In [ ]:
result = model.predict(source="/content/021745.jpg")
for r in result:
    r.show()

In [ ]:
model.save()

In [ ]:
from ultralytics import YOLO
import cv2
from google.colab.patches import cv2_imshow
from IPython.display import display, Image

# Load your trained YOLOv8 model
model = YOLO('/content/computer_vision_model.pt')

# Open the video file
cap = cv2.VideoCapture('/content/est_v2.mov')

# Optional: setup video writer to save output
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps = cap.get(cv2.CAP_PROP_FPS)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter('output_video.mp4', fourcc, fps, (frame_width, frame_height))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLOv8 detection on the frame
    results = model.predict(source=frame, imgsz=640, conf=0.4, verbose=False)

    # results[0].plot() returns a numpy array with bounding boxes drawn
    annotated_frame = results[0].plot()

    # Using display and Image to display the annotated frame
    #display(Image(data=cv2.imencode('.jpg', annotated_frame)[1].tobytes()))


    # Write the frame into the output video
    out.write(annotated_frame)

    # Press 'q' to exit early
    #if cv2.waitKey(1) & 0xFF == ord('q'):
    #    break

# Release everything
cap.release()
out.release()
cv2.destroyAllWindows()

In [ ]:
# Import necessary libraries
from ultralytics import YOLO
import cv2
import time # Used to calculate elapsed time and control printing frequency
import numpy as np # Needed for text background
import os # Needed for file path operations

# Define the zone boundaries using the provided rectangular coordinates (top-left, bottom-right)
# These coordinates are based on a 1024x1024 resolution as discussed.
zones = {
    "waiting_north": ((435, 0), (510, 385)),
    "waiting_west": ((0, 510), (385, 590)),
    "waiting_east": ((635, 435), (1024, 510)),
    "waiting_south": ((510, 635), (590, 1024)),
    "intersection": ((385, 385), (635, 635))
}

# Define the mapping from class ID (integer) to vehicle type name
# IMPORTANT: You MUST adjust this mapping based on the actual class IDs
# output by YOUR trained YOLOv8 model. Refer to your dataset.yaml names list.
# This is a placeholder mapping based on common vehicle types.
class_id_to_type = {
    0: "City Car",
    1: "Bus",
    2: "Tram",
    3: "Taxi",
    4: "Police Car",
    5: "Ambulance",
    6: "Delivery Van",
    7: "Semi Truck",
    8: "Road Maintenance Vehicle",
    9: "Park Maintenance Vehicle",
    10: "Hearse",
    11: "City Car", # Example: if 'City Car' has multiple IDs
    12: "Garbage Truck",
    13: "Post Van",
    14: "Motorcycle",
    15: "Trailer",
    # Add mappings for Hatchback/Sedan, Van, Muscle Car if they are separate classes
    # e.g., 16: "Hatchback/Sedan", 17: "Van", 18: "Muscle Car"
}

# Define priority values based on vehicle type name and your updated rules
# Mapping vehicle type names (from the image and your grouping) to their priority values
vehicle_type_priority = {
    "City Car": 5,
    "Motorcycle": 2,
    "Road Maintenance Vehicle": 2,
    "Park Maintenance Vehicle": 2,
    "Hearse": 2,
    "Muscle Car": 5,
    "Taxi": 5,
    "Hatchback/Sedan": 5, # Assuming this type exists and maps to a class ID
    "Van": 5, # Assuming this type exists and maps to a class ID
    "Bus": 80,
    "Tram": 240,
    "Police Car": float('inf'), # High priority for emergency vehicles
    "Ambulance": float('inf'), # High priority for emergency vehicles
    "Delivery Van": 4,
    "Semi Truck": 14.5,
    "Post Van": 2,
    "Garbage Truck": 15,
    "Trailer": 0,
    # Add priority for Firetruck if it's a distinct class
    "Firetruck": float('inf') # High priority for emergency vehicles
}

# Define passenger capacity values based on vehicle type name (from the image)
vehicle_type_passenger_capacity = {
    "City Car": 2,
    "Motorcycle": 2,
    "Road Maintenance Vehicle": 2,
    "Park Maintenance Vehicle": 2,
    "Hearse": 2,
    "Muscle Car": 4,
    "Taxi": 5,
    "Hatchback/Sedan": 5,
    "Van": 5,
    "Bus": 80,
    "Tram": 240,
    "Delivery Van": 2,
    "Semi Truck": 2,
    "Post Van": 1,
    "Garbage Truck": 5,
    "Trailer": 0, # Trailers don't carry passengers
    "Police Car": 0, # Assuming passenger count is not relevant for priority calculation here
    "Ambulance": 0, # Assuming passenger count is not relevant for priority calculation here
    "Firetruck": 0 # Assuming passenger count is not relevant for priority calculation here
}


# Function to get priority based on class ID
def get_priority(class_id):
    """Looks up the priority value for a given class ID."""
    vehicle_name = class_id_to_type.get(class_id, "Unknown") # Default to "Unknown" if ID not in map
    return vehicle_type_priority.get(vehicle_name, 0) # Default to 0 priority for unknown types


# Function to get passenger count based on class ID
def get_passenger_count(class_id):
    """Looks up the passenger count for a given class ID."""
    vehicle_name = class_id_to_type.get(class_id, "Unknown") # Default to "Unknown" if ID not in map
    # Return passenger count, default to 0 if type not in capacity map or is an emergency/trailer type
    return vehicle_type_passenger_capacity.get(vehicle_name, 0)


# Load your trained YOLOv8 model
# Update the path if your model is saved elsewhere
try:
    model = YOLO('/content/runs/detect/train/weights/last.pt')
    print("YOLOv8 model loaded successfully.")
except Exception as e:
    print(f"Error loading YOLOv8 model: {e}")
    print("Please ensure the model path is correct and the model file exists.")
    # Exit or handle the error appropriately
    exit()


# Open the input video file
video_path = '/content/est_v2.mov' # Update this path to your input video file
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print(f"Error: Could not open input video file {video_path}")
    # Exit or handle the error appropriately
    exit()

# Get video properties
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"Processing video: {video_path}")
print(f"Resolution: {frame_width}x{frame_height}")
print(f"FPS: {fps}")
print(f"Total frames: {frame_count}")

# --- Set up VideoWriter to save the output video ---
output_video_path = 'output_video_with_analysis.mp4' # Name of the output file
# Define the codec and create VideoWriter object
# 'mp4v' is a common codec, but you might need to try others ('XVID', 'MJPG', 'DIVX')
# depending on your system and desired output format.
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

if not out.isOpened():
    print(f"Error: Could not create output video file {output_video_path}")
    # Handle the error, potentially exit or continue without saving
    # For this example, we'll print an error and continue without saving
    out = None # Set to None to indicate writing is not possible


# Calculate the number of frames to process before updating the overlay
# This is set to roughly match the video's FPS for updating every second
frames_to_analyze_per_overlay_update = max(1, int(fps)) # Ensure at least 1 frame per update

frame_counter = 0
print(f"\nAnalyzing zones and overlaying results...")
if out is not None:
    print(f"Saving output to: {output_video_path}")


# Create a window to display the video (optional, can be removed if only saving)
# cv2.namedWindow('Video Feed with Zone Analysis', cv2.WINDOW_NORMAL) # Use WINDOW_NORMAL for resizable window

# Loop through the video frames
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break # End of video

    frame_counter += 1

    # Perform YOLOv8 detection on the frame
    # Using conf=0.4 as in the previous cell, verbose=False to keep output clean
    results = model.predict(source=frame, imgsz=640, conf=0.4, verbose=False)

    # Get the detected bounding boxes and class IDs for the current frame
    detections = results[0].boxes # Access the Boxes object containing detections

    # Dictionary to hold vehicles detected in each zone for the current frame
    vehicles_in_zones_current_frame = {zone_name: [] for zone_name in zones.keys()}

    # Check if any objects were detected in the frame
    if detections is not None and detections.xyxy is not None:
        # Iterate through each detected object's bounding box
        for i in range(len(detections.xyxy)):
            # Get bounding box coordinates (x1, y1, x2, y2)
            x1, y1, x2, y2 = detections.xyxy[i].tolist()
            # Get class ID (integer)
            class_id = int(detections.cls[i].item())
            # Get confidence score (float)
            confidence = detections.conf[i].item()

            # Calculate the center point of the bounding box
            center_x = (x1 + x2) / 2
            center_y = (y1 + y2) / 2

            # Check which zone the center point of the vehicle falls into
            for zone_name, ((zx1, zy1), (zx2, zy2)) in zones.items():
                # Check if the center point (center_x, center_y) is within the rectangular zone
                # A point (px, py) is inside a rectangle defined by (rx1, ry1) and (rx2, ry2)
                # if rx1 <= px <= rx2 and ry1 <= py <= ry2
                if zx1 <= center_x <= zx2 and zy1 <= center_y <= zy2:
                    # Append the detected vehicle info to the list for this zone
                    vehicles_in_zones_current_frame[zone_name].append({
                        "class_id": class_id,
                        "confidence": confidence,
                        "priority": get_priority(class_id), # Get the priority for this vehicle
                        "passenger_count": get_passenger_count(class_id) # Get the passenger count
                        # You could add more info here, like the bounding box itself:
                        # "bbox": (x1, y1, x2, y2)
                        # Or track_id if using a tracking model:
                        # "track_id": detections.id[i].item() if detections.id is not None else None
                    })
                    # Assuming a vehicle is only in one zone at a time for simplicity.
                    # If zones can overlap and you want to count in multiple, remove the 'break'
                    break # Move to the next detected vehicle after finding its zone

    # --- Overlay Analysis Results on the Frame ---

    # Draw detected bounding boxes on the frame
    annotated_frame = results[0].plot() # This draws the YOLO detections

    # Draw zone rectangles and overlay analysis text
    for zone_name, ((zx1, zy1), (zx2, zy2)) in zones.items():
        # Define color for the zone rectangle (BGR format)
        color = (0, 255, 0) # Green

        # Draw the rectangular zone outline
        cv2.rectangle(annotated_frame, (zx1, zy1), (zx2, zy2), color, 2)

        # Calculate analysis metrics for the current zone
        zone_vehicles = vehicles_in_zones_current_frame[zone_name]
        num_vehicles = len(zone_vehicles)
        total_priority = sum(v['priority'] for v in zone_vehicles)
        total_passengers = sum(v['passenger_count'] for v in zone_vehicles)

        # Prepare text to display
        text_lines = [
            f"{zone_name}: {num_vehicles} vehicles",
            f"Priority: {total_priority:.2f}",
            f"Passengers: {total_passengers}"
        ]

        # Define text properties
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.6
        font_thickness = 1
        text_color = (255, 255, 255) # White
        background_color = (0, 0, 0) # Black

        # Position the text near the top-left corner of the zone
        text_x = zx1 + 5
        text_y = zy1 + 20 # Starting position for the first line

        # Overlay each line of text
        for i, line in enumerate(text_lines):
            # Get text size to create a background rectangle
            (text_width, text_height), baseline = cv2.getTextSize(line, font, font_scale, font_thickness)

            # Draw background rectangle for text
            cv2.rectangle(annotated_frame, (text_x, text_y - text_height - baseline),
                          (text_x + text_width, text_y + baseline), background_color, -1)

            # Put the text on the frame
            cv2.putText(annotated_frame, line, (text_x, text_y), font, font_scale, text_color, font_thickness)

            # Move down for the next line
            text_y += text_height + baseline + 5 # Add some spacing between lines

    # --- Write the frame to the output video file ---
    if out is not None:
        try:
            out.write(annotated_frame)
        except Exception as e:
            print(f"Error writing frame {frame_counter} to video file: {e}")
            # Stop writing if an error occurs
            out.release()
            out = None # Set to None to prevent further writing attempts


    # --- Display the frame in the window (optional) ---
    # If you want to see the video processing in real-time, uncomment the lines below.
    # If you only want to save the output file, keep these commented out.
    # cv2.imshow('Video Feed with Zone Analysis', annotated_frame)

    # # Break the loop if 'q' is pressed (only works if cv2.imshow is active)
    # if cv2.waitKey(1) & 0xFF == ord('q'):
    #     break


# --- Clean up resources ---
cap.release()
if out is not None:
    out.release() # Release the VideoWriter
# If cv2.imshow was used, uncomment the line below to close windows
# cv2.destroyAllWindows()

print("\nVideo processing finished.")
if os.path.exists(output_video_path):
    print(f"Output video saved as: {output_video_path}")
else:
    print("Output video file was not created or saved successfully.")



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
